## First stage: Claim Extractor

This first stage is already done by the LLM's pipeline, we will have to evaluate the outputs of the models Qwen3-1.7b (RAG) and Qwen3-4b (LoRA and RAG)
the output is the the folder Knowledge graph, in this case on the folder doc_146120936:
- LoRA-Output-4B_doc_146120936.json (Qwen3-4b)
- RAG-Output-4B_doc_146120936.json (Qwen3-4b)
- RAG-Output_doc_146120936.json (Qwen3-1.7b)

In [20]:
import glob

import json

import os
from huggingface_hub import InferenceClient
from groq import Groq
import time

from tqdm import tqdm

import random

In [21]:
GROQ_KEY = os.getenv("GROQ_API_KEY")


In [22]:
FILES = glob.glob("./results/clean_*-Output*.json")
gold_qa = "./Knowledge-graph/doc_146120936/Gold-QA_146120936.json"

random.seed(42)

with open(gold_qa, "r", encoding="utf-8") as f:
    gold_data = json.load(f)
    gold_dict = {item["question"]: item.get("triplets") for item in gold_data["pairs"]}

total_questions = 0
for file in FILES:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
        total_questions += len(data["results"])

print(f"Total questions to process: {total_questions}")

Answers = []
num_questions = 0

progress = tqdm(total=total_questions, desc="Global Progress", unit="q")

client = Groq(api_key=GROQ_KEY)

for file in FILES:
    print("Processing file:", file)

    os.makedirs("results", exist_ok=True)

    output_file = os.path.join("results", f"evaluated_{os.path.basename(file)}")

    results = {"results": []}

    with open(file, "r", encoding="utf-8") as f:

        data = json.load(f)
        for item in data["results"]:
            question = item["question"]

            num_questions += 1

            t = item["triplets"]

            if isinstance(t, dict) and "triplets" in t:
                triplets = t["triplets"]
            elif isinstance(t, list):
                triplets = t
            else:
                triplets = []

            gold_answer = gold_dict.get(question, None)

            prompt = f"""
You are verifying whether extracted knowledge triplets correctly answer a scientific question.

You have:
- <QUESTION>
- <TRIPLETS>
- <GOLD_STANDARD_ANSWER>

Return ONLY:
0 - supported
1 - contradicted
2 - not verifiable

<QUESTION>
{question}
</QUESTION>

<TRIPLETS>
{triplets}
</TRIPLETS>

<GOLD_STANDARD_ANSWER>
{gold_answer}
</GOLD_STANDARD_ANSWER>

Answer only with a single number: 0, 1, or 2.
"""

            completion = client.chat.completions.create(
                model="groq/compound",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )
            result = completion.choices[0].message.content

            output_row = {
                "question": question,
                "triplets": triplets,
                "gold_answer": gold_answer,
                "result": result,
            }
            
            results["results"].append(output_row)

            with open(output_file, "w", encoding="utf-8") as f_out:
                json.dump(results, f_out, indent=4, ensure_ascii=False)

            print(f"\nSaved output to: {output_file}\n")

            progress.update(1)

            time.sleep(5)
            
    
print("Total questions processed:", num_questions)

Total questions to process: 147


Global Progress:   0%|          | 0/147 [00:00<?, ?q/s]

Processing file: ./results\clean_LoRA-Output-4B_doc_146120936.json


Global Progress:   1%|          | 1/147 [00:09<22:08,  9.10s/q]


Saved output to: results\evaluated_clean_LoRA-Output-4B_doc_146120936.json



...


Saved output to: results\evaluated_clean_RAG-Output_doc_146120936.json



Global Progress:  99%|█████████▉| 146/147 [20:05<00:08,  8.59s/q]


Saved output to: results\evaluated_clean_RAG-Output_doc_146120936.json



Global Progress: 100%|██████████| 147/147 [20:13<00:00,  8.16s/q]


Saved output to: results\evaluated_clean_RAG-Output_doc_146120936.json

Total questions processed: 147


In [ ]:
json_example = {
    "results": [
        {
            "question": "What is another name for Content-Aware ReAssembly of FEatures?",
            "triplets": {
                "triplets": [
                    "[subject:Content-Aware ReAssembly of FEatures, Synonym-Of, CARAFE]"
                ]
            }
        }
    ]
}

Answers = []
num_questions = 0


for result in json_example["results"]:
    question = result["question"]

    num_questions += 1

    t = result["triplets"]

    # Case 1: {"triplets": [...], "raw_output": "..."}
    if isinstance(t, dict) and "triplets" in t:
        triplets = t["triplets"]

    # Case 2: ["[subject:..., ...]"]
    elif isinstance(t, list):
        triplets = t

    # Fallback
    else:
        triplets = []

    print("Question:", question)
    print("Triplets:", triplets)
    

prompt = f"""
You are a helpful assistant that will verify the answer to a question based on provided knowledge triplets.

You will be provided with:
1. A question.
2. A set of knowledge triplets in the format: [subject:..., predicate:..., object:...].
3. A gold standard answer to the question.

You can answer in 3 ways:
- Supported — the triplet matches the information in the gold standard;
- Contradicted — the triplet conflicts with the gold standard;
- Not verifiable — the triplet cannot be confirmed using the available information.

<QUESTION>
{question}
</QUESTION>
<TRIPLETS>
{triplets}
</TRIPLETS>
<GOLD_STANDARD_ANSWER>
["CARAFE:Method","Synonym-Of","Content - Aware ReAssembly of FEatures:Method"],
</GOLD_STANDARD_ANSWER>

output only:
0 - supoported
1 - contradicted
2 - not verifiable
""" 

client = Groq(api_key=GROQ_KEY)
completion = client.chat.completions.create(
    model="groq/compound",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print(completion.choices[0].message.content)

Question: What is another name for Content-Aware ReAssembly of FEatures?
Triplets: ['[subject:Content-Aware ReAssembly of FEatures, Synonym-Of, CARAFE]']
0
